# Complete PRE/POST qEEG response-association grid

## Objective

The published response analysis used four selected condition/state/feature cells. This notebook fills the complete grid: **PRE and POST × awake and sleep × DFA, entropy, and PLI**. For every cell, it tests both one clip at a time and the mean of the two matching clips from each patient.

The goal is to show every measured association, including negative results, rather than report only the combinations selected in the source paper.

In [1]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

from IPython.display import display

SEED = 20260912
PERMUTATIONS = 1_000_000
REPO = Path(os.environ.get("IESSEEG_BASELINES_REPO", Path.cwd())).resolve()
METADATA_CSV = Path(os.environ["IESSEEG_METADATA_CSV"])
RAW_FEATURES_DIR = Path(os.environ["IESSEEG_RAJARAMAN_RAW_FEATURES_DIR"])

if not METADATA_CSV.is_file():
    raise FileNotFoundError("IESSEEG_METADATA_CSV does not point to a file")
if not RAW_FEATURES_DIR.is_dir():
    raise NotADirectoryError("IESSEEG_RAJARAMAN_RAW_FEATURES_DIR does not point to a directory")

analysis_dir = REPO / "analysis" / "response_features"
sys.path.insert(0, str(analysis_dir))
run_analysis = importlib.import_module("analyze_full_qeeg_response_grid").run_analysis

print(f"Configuration loaded: {PERMUTATIONS:,} patient-level permutations; seed {SEED}.")

Configuration loaded: 1,000,000 patient-level permutations; seed 20260912.


## Experimental design

Each condition/state cell contains two clips from each of 50 patients. Immediate response contains 32 responders and 18 non-responders; sustained response contains 28 responders and 22 non-responders.

For a single-clip test, both clips contribute values, but the significance test keeps the two values from one patient together. For a patient-average test, the two matching clips are averaged first. The raw P-value tests whether response labels and EEG values are unrelated across patients. Holm correction covers all 24 grid tests within each endpoint.

In [2]:
grid = run_analysis(
    METADATA_CSV,
    RAW_FEATURES_DIR,
    n_permutations=PERMUTATIONS,
    seed=SEED,
)
assert len(grid) == 48
assert set(grid.groupby("endpoint").size()) == {24}
print("Complete grid verified: 24 tests for each response endpoint.")

Complete grid verified: 24 tests for each response endpoint.


## Results

`Separation AUROC` is always at least 0.5 and shows the strength of separation in the better direction. `Direction` states whether responders have higher or lower values. The final column is the decision after correcting all 24 comparisons for that endpoint.

In [3]:
columns = {
    "input": "EEG",
    "quantity": "Quantity",
    "aggregation": "Calculation unit",
    "source_selected_cell": "Used by source paper",
    "separation_auc": "Separation AUROC",
    "responder_direction": "Direction",
    "patient_clustered_permutation_p": "Raw P",
    "holm_p_across_24_grid_tests": "Holm P",
    "significant_after_holm_0_05": "Significant",
}
for endpoint in ("immediate", "sustained"):
    print(f"{endpoint.title()} response")
    view = grid.loc[grid.endpoint.eq(endpoint), list(columns)].rename(columns=columns)
    display(
        view.style.format(
            {
                "Separation AUROC": "{:.3f}",
                "Raw P": "{:.6g}",
                "Holm P": "{:.6g}",
            }
        )
    )

Immediate response


,EEG,Quantity,Calculation unit,Used by source paper,Separation AUROC,Direction,Raw P,Holm P,Significant
0,PRE awake,beta DFA intercept,one clip,True,0.566,lower,0.411669,1,False
1,PRE awake,beta DFA intercept,mean of two awake clips,True,0.587,lower,0.319591,1,False
2,PRE awake,beta Shannon entropy,one clip,False,0.557,higher,0.463058,1,False
3,PRE awake,beta Shannon entropy,mean of two awake clips,False,0.562,higher,0.476341,1,False
4,PRE awake,delta PLI connectivity,one clip,True,0.614,lower,0.153661,1,False
5,PRE awake,delta PLI connectivity,mean of two awake clips,True,0.607,lower,0.218102,1,False
6,PRE sleep,beta DFA intercept,one clip,False,0.505,lower,0.95624,1,False
7,PRE sleep,beta DFA intercept,mean of two sleep clips,False,0.500,higher,1,1,False
8,PRE sleep,beta Shannon entropy,one clip,False,0.653,higher,0.0506599,1,False
9,PRE sleep,beta Shannon entropy,mean of two sleep clips,False,0.661,higher,0.0596659,1,False


Sustained response


,EEG,Quantity,Calculation unit,Used by source paper,Separation AUROC,Direction,Raw P,Holm P,Significant
24,PRE awake,beta DFA intercept,one clip,True,0.655,lower,0.042051,0.756917,False
25,PRE awake,beta DFA intercept,mean of two awake clips,True,0.683,lower,0.026276,0.525519,False
26,PRE awake,beta Shannon entropy,one clip,False,0.548,higher,0.522764,1,False
27,PRE awake,beta Shannon entropy,mean of two awake clips,False,0.544,higher,0.607329,1,False
28,PRE awake,delta PLI connectivity,one clip,True,0.666,lower,0.029587,0.562152,False
29,PRE awake,delta PLI connectivity,mean of two awake clips,True,0.657,lower,0.0567649,0.965004,False
30,PRE sleep,beta DFA intercept,one clip,False,0.615,lower,0.149721,1,False
31,PRE sleep,beta DFA intercept,mean of two sleep clips,False,0.615,lower,0.169835,1,False
32,PRE sleep,beta Shannon entropy,one clip,False,0.638,higher,0.0675809,1,False
33,PRE sleep,beta Shannon entropy,mean of two sleep clips,False,0.636,higher,0.102452,1,False


## Which results survive correction?

The table below contains only associations that remain below 0.05 after accounting for all 24 displayed grid tests. This filter is for reading convenience; the complete tables above remain the primary result.

In [4]:
significant = grid.loc[
    grid.significant_after_holm_0_05,
    [
        "endpoint",
        "input",
        "quantity",
        "aggregation",
        "source_selected_cell",
        "separation_auc",
        "responder_direction",
        "holm_p_across_24_grid_tests",
    ],
]
display(significant.reset_index(drop=True))

,endpoint,input,quantity,aggregation,source_selected_cell,separation_auc,responder_direction,holm_p_across_24_grid_tests
0,immediate,POST sleep,beta Shannon entropy,one clip,True,0.781250,higher,0.001920
1,immediate,POST sleep,beta Shannon entropy,mean of two sleep clips,True,0.822917,higher,0.002438
2,sustained,POST awake,beta DFA intercept,one clip,True,0.768669,lower,0.003024
3,sustained,POST awake,beta DFA intercept,mean of two awake clips,True,0.816558,lower,0.001650
4,sustained,POST sleep,beta Shannon entropy,one clip,True,0.802760,higher,0.000168
5,sustained,POST sleep,beta Shannon entropy,mean of two sleep clips,True,0.852273,higher,0.000230


## Complete tables: individual and paper-defined combined metrics

The four tables below contain **all individual and combined results together**, divided only by response endpoint and calculation level. Each table has 20 rows: 12 individual qEEG results and 8 paper-derived results. The paper-derived quantities are R0, duration-adjusted P0, R1, duration-adjusted P1, PRE-to-POST changes in awake DFA, awake PLI, and sleep entropy, and the relapse metric rho.

A clip-based combined result uses one PRE clip, a within-patient POST awake/sleep clip pairing, or a within-patient PRE/POST pairing as required by its formula. A patient-based result first averages the two matching clips for that patient. R0 and R1 contain EEG only. P0 and P1 add the clinical duration category. Rho was defined for time to relapse, not binary response; its response association is displayed only to make the catalog complete and is labeled with its original target.

In [5]:
derived_module = importlib.import_module("analyze_paper_derived_response_metrics")
derived = derived_module.run_analysis(
    METADATA_CSV,
    RAW_FEATURES_DIR,
    n_permutations=PERMUTATIONS,
    seed=20260914,
)
catalog = derived_module.build_complete_catalog(grid, derived)
assert len(derived) == 32
assert len(catalog) == 80
print("Complete catalog verified: 48 individual-feature rows plus 32 paper-derived rows.")

Complete catalog verified: 48 individual-feature rows plus 32 paper-derived rows.


In [6]:
complete_columns = {
    "metric_category": "Metric type",
    "input": "EEG input",
    "quantity": "Metric",
    "aggregation": "How it was calculated",
    "n_values": "Values",
    "responder_median": "Responder median",
    "nonresponder_median": "Non-responder median",
    "separation_auc": "Separation AUROC",
    "responder_direction": "Responder direction",
    "raw_p": "Raw P",
    "holm_adjusted_p": "Holm P",
    "adjustment_family": "Correction family",
}
for endpoint in ("immediate", "sustained"):
    for level in ("clip-based", "patient-based"):
        print(f"{endpoint.title()} response — {level}: all individual and combined metrics")
        view = catalog.loc[
            catalog.endpoint_tested_here.eq(endpoint)
            & catalog.calculation_level.eq(level),
            list(complete_columns),
        ].rename(columns=complete_columns)
        assert len(view) == 20
        display(
            view.style.format(
                {
                    "Responder median": "{:.4g}",
                    "Non-responder median": "{:.4g}",
                    "Separation AUROC": "{:.3f}",
                    "Raw P": "{:.6g}",
                    "Holm P": "{:.6g}",
                }
            )
        )

Immediate response — clip-based: all individual and combined metrics


,Metric type,EEG input,Metric,How it was calculated,Values,Responder median,Non-responder median,Separation AUROC,Responder direction,Raw P,Holm P,Correction family
0,individual qEEG feature,POST awake,beta DFA intercept,one clip,100,-0.4083,-0.1644,0.689,lower,0.011309,0.237489,24 individual-feature tests within endpoint
1,individual qEEG feature,POST awake,beta Shannon entropy,one clip,100,5.916,5.845,0.527,higher,0.740673,1,24 individual-feature tests within endpoint
2,individual qEEG feature,POST awake,delta PLI connectivity,one clip,100,7.018,10.82,0.590,lower,0.26785,1,24 individual-feature tests within endpoint
3,individual qEEG feature,POST sleep,beta DFA intercept,one clip,100,-0.4635,-0.3848,0.564,lower,0.42798,1,24 individual-feature tests within endpoint
4,individual qEEG feature,POST sleep,beta Shannon entropy,one clip,100,5.918,5.385,0.781,higher,7.99999e-05,0.00192,24 individual-feature tests within endpoint
5,individual qEEG feature,POST sleep,delta PLI connectivity,one clip,100,10.23,21.05,0.571,lower,0.385352,1,24 individual-feature tests within endpoint
6,individual qEEG feature,PRE awake,beta DFA intercept,one clip,100,-0.1608,-0.09431,0.566,lower,0.411669,1,24 individual-feature tests within endpoint
7,individual qEEG feature,PRE awake,beta Shannon entropy,one clip,100,5.942,5.794,0.557,higher,0.463058,1,24 individual-feature tests within endpoint
8,individual qEEG feature,PRE awake,delta PLI connectivity,one clip,100,5.848,12.87,0.614,lower,0.153661,1,24 individual-feature tests within endpoint
9,individual qEEG feature,PRE sleep,beta DFA intercept,one clip,100,-0.03438,-0.03092,0.505,lower,0.95624,1,24 individual-feature tests within endpoint


Immediate response — patient-based: all individual and combined metrics


,Metric type,EEG input,Metric,How it was calculated,Values,Responder median,Non-responder median,Separation AUROC,Responder direction,Raw P,Holm P,Correction family
20,individual qEEG feature,POST awake,beta DFA intercept,mean of two awake clips,50,-0.3903,-0.09649,0.731,lower,0.00651399,0.143308,24 individual-feature tests within endpoint
21,individual qEEG feature,POST awake,beta Shannon entropy,mean of two awake clips,50,5.931,5.763,0.538,higher,0.667595,1,24 individual-feature tests within endpoint
22,individual qEEG feature,POST awake,delta PLI connectivity,mean of two awake clips,50,9.942,7.895,0.589,lower,0.30304,1,24 individual-feature tests within endpoint
23,individual qEEG feature,POST sleep,beta DFA intercept,mean of two sleep clips,50,-0.4817,-0.3682,0.564,lower,0.464983,1,24 individual-feature tests within endpoint
24,individual qEEG feature,POST sleep,beta Shannon entropy,mean of two sleep clips,50,5.932,5.253,0.823,higher,0.000106,0.002438,24 individual-feature tests within endpoint
25,individual qEEG feature,POST sleep,delta PLI connectivity,mean of two sleep clips,50,12.13,22.51,0.569,lower,0.427738,1,24 individual-feature tests within endpoint
26,individual qEEG feature,PRE awake,beta DFA intercept,mean of two awake clips,50,-0.1821,-0.07353,0.587,lower,0.319591,1,24 individual-feature tests within endpoint
27,individual qEEG feature,PRE awake,beta Shannon entropy,mean of two awake clips,50,5.826,5.709,0.562,higher,0.476341,1,24 individual-feature tests within endpoint
28,individual qEEG feature,PRE awake,delta PLI connectivity,mean of two awake clips,50,6.287,11.84,0.607,lower,0.218102,1,24 individual-feature tests within endpoint
29,individual qEEG feature,PRE sleep,beta DFA intercept,mean of two sleep clips,50,-0.05444,-0.09045,0.500,higher,1,1,24 individual-feature tests within endpoint


Sustained response — clip-based: all individual and combined metrics


,Metric type,EEG input,Metric,How it was calculated,Values,Responder median,Non-responder median,Separation AUROC,Responder direction,Raw P,Holm P,Correction family
40,individual qEEG feature,POST awake,beta DFA intercept,one clip,100,-0.4399,-0.0965,0.769,lower,0.000144,0.003024,24 individual-feature tests within endpoint
41,individual qEEG feature,POST awake,beta Shannon entropy,one clip,100,5.905,5.955,0.507,lower,0.925351,1,24 individual-feature tests within endpoint
42,individual qEEG feature,POST awake,delta PLI connectivity,one clip,100,8.187,8.48,0.562,lower,0.428692,1,24 individual-feature tests within endpoint
43,individual qEEG feature,POST sleep,beta DFA intercept,one clip,100,-0.475,-0.3443,0.606,lower,0.171057,1,24 individual-feature tests within endpoint
44,individual qEEG feature,POST sleep,beta Shannon entropy,one clip,100,5.988,5.413,0.803,higher,6.99999e-06,0.000168,24 individual-feature tests within endpoint
45,individual qEEG feature,POST sleep,delta PLI connectivity,one clip,100,8.187,21.05,0.617,lower,0.135386,1,24 individual-feature tests within endpoint
46,individual qEEG feature,PRE awake,beta DFA intercept,one clip,100,-0.2107,0.01732,0.655,lower,0.042051,0.756917,24 individual-feature tests within endpoint
47,individual qEEG feature,PRE awake,beta Shannon entropy,one clip,100,5.934,5.853,0.548,higher,0.522764,1,24 individual-feature tests within endpoint
48,individual qEEG feature,PRE awake,delta PLI connectivity,one clip,100,5.263,12.87,0.666,lower,0.029587,0.562152,24 individual-feature tests within endpoint
49,individual qEEG feature,PRE sleep,beta DFA intercept,one clip,100,-0.08595,0.1059,0.615,lower,0.149721,1,24 individual-feature tests within endpoint


Sustained response — patient-based: all individual and combined metrics


,Metric type,EEG input,Metric,How it was calculated,Values,Responder median,Non-responder median,Separation AUROC,Responder direction,Raw P,Holm P,Correction family
60,individual qEEG feature,POST awake,beta DFA intercept,mean of two awake clips,50,-0.4423,-0.08665,0.817,lower,7.49999e-05,0.00165,24 individual-feature tests within endpoint
61,individual qEEG feature,POST awake,beta Shannon entropy,mean of two awake clips,50,5.902,5.935,0.503,lower,0.976833,1,24 individual-feature tests within endpoint
62,individual qEEG feature,POST awake,delta PLI connectivity,mean of two awake clips,50,9.942,7.895,0.569,lower,0.409756,1,24 individual-feature tests within endpoint
63,individual qEEG feature,POST sleep,beta DFA intercept,mean of two sleep clips,50,-0.5133,-0.3682,0.607,lower,0.19851,1,24 individual-feature tests within endpoint
64,individual qEEG feature,POST sleep,beta Shannon entropy,mean of two sleep clips,50,5.973,5.375,0.852,higher,9.99999e-06,0.00023,24 individual-feature tests within endpoint
65,individual qEEG feature,POST sleep,delta PLI connectivity,mean of two sleep clips,50,10.96,22.51,0.621,lower,0.146358,1,24 individual-feature tests within endpoint
66,individual qEEG feature,PRE awake,beta DFA intercept,mean of two awake clips,50,-0.2083,-0.04535,0.683,lower,0.026276,0.525519,24 individual-feature tests within endpoint
67,individual qEEG feature,PRE awake,beta Shannon entropy,mean of two awake clips,50,5.826,5.772,0.544,higher,0.607329,1,24 individual-feature tests within endpoint
68,individual qEEG feature,PRE awake,delta PLI connectivity,mean of two awake clips,50,5.848,12.43,0.657,lower,0.0567649,0.965004,24 individual-feature tests within endpoint
69,individual qEEG feature,PRE sleep,beta DFA intercept,mean of two sleep clips,50,-0.09131,0.1714,0.615,lower,0.169835,1,24 individual-feature tests within endpoint


## Interpretation boundary

The four cells marked as source-selected reproduce quantities chosen in the published sustained-response analysis. The other eight feature cells are newly examined here. The 24 individual-feature tests and 16 paper-derived tests are corrected as separate, explicitly named families. Rho remains a relapse-time metric even when its numerical association with a response label is displayed. Because the locally reproduced entropy has a constant absolute offset from the paper, P1 is useful here for ranking and association but its numerical probability is not calibrated. All results use the source cohort and do not estimate performance on a new clinical cohort.